# Neural Networks with TensorFlow and Keras
## Neural network exercises and Google stock price prediction

This notebook develops neural network models of varying complexity using Keras and TensorFlow.
All exercise answers are included in the notebook, either as completed code or written explanations.

*Original exercise template: version 1, updated on November 20, 2020.*
*English translation: saved execution outputs have been cleared. The original modeling approach has not been corrected; see README.md for limitations.*

### Instructions

- Complete the exercises individually.
- Submit this notebook with the solutions to the exercises.
- The original assignment requested submission through the Instituto BME platform before the deadline.
- Include your first and last names in the following code block.

In [ ]:
# TO-DO: Enter your first and last names here:
student_data = {'first_name': 'Luis Alberto', 'last_name': 'Román Alcaide'}

### Accessing the exercise materials

The easiest way to access all the exercise materials is to clone the following GitHub repository:

In [ ]:
!git clone https://github.com/luisferuam/MIAX5

### Importing libraries

The following cell imports the libraries required for the exercises. Add any additional imports at the end.

In [ ]:
%tensorflow_version 2.x
import tensorflow as tf
from tensorflow import keras

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import sys
sys.path.append('MIAX5')
import dlfbt
# Add any additional imports here

### Part I: Simple models with Keras

The first models use synthetic data. The following cell creates a one-dimensional regression problem and plots `t` against `x`:

In [ ]:
dg = dlfbt.DataGeneratorLinear1D()
dg.create_dataset(seed=21)

plt.figure(figsize=(5, 5))
plt.plot(dg.x, dg.t, '.')
plt.grid(True)
plt.xlabel('x')
plt.ylabel('t')
plt.show()

print("Shape of x:", dg.x.shape)
print("Shape of t:", dg.t.shape)

#### Exercise 1 [1 point]

Build a linear regression model in Keras to predict `t` from `x`.

**[1.1]** Define the model by completing the following cell:

In [ ]:
model = keras.Sequential()
#-------------------------------------------------------------
model.add(keras.layers.Dense(100,input_shape=(1,)))
#-------------------------------------------------------------
model.summary()

**[1.2]** Compile the model using SGD and an appropriate loss function. Adjust the learning rate so that training converges:

In [ ]:

#-------------------------------------------------------------
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01),
              loss='mse')
#-------------------------------------------------------------

The following cell fits the model to the data and plots the results:

In [ ]:
history = model.fit(dg.x, dg.t, epochs=1000, batch_size=dg.x.shape[0], verbose=0)
y = model.predict(dg.x)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'])
plt.grid(True)
plt.xlabel('epoch')
plt.ylabel('MSE')

plt.subplot(1, 2, 2)
plt.plot(dg.x, dg.t, '.')
plt.plot(dg.x, y, 'r-')
plt.grid(True)
plt.xlabel('x')
plt.ylabel('t')
plt.legend(('real', 'predicted'))
plt.show()

Next, consider a logistic regression problem. The following cell generates two-dimensional data and plots the distribution of `x` for the two classes, `t=0` and `t=1`:

In [ ]:
dg = dlfbt.DataGeneratorLogistic(a=[2.0, 2.0], b=-20.0)
dg.create_dataset(n=500, seed=17)

plt.figure(figsize=(5, 5))
ix0 = (dg.t.ravel() == 0)
plt.plot(dg.x[ix0, 0], dg.x[ix0, 1], '.', label='t=0')
ix1 = (dg.t.ravel() == 1)
plt.plot(dg.x[ix1, 0], dg.x[ix1, 1], '.', label='t=1')
plt.grid(True)
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.show()

print("Shape of x:", dg.x.shape)
print("Shape of t:", dg.t.shape)

#### Exercise 2 [1 point]

Build a logistic regression model in Keras to predict `t` from `x`.

**[2.1]** Define the model by completing the following cell:

In [ ]:
model = keras.Sequential()

#-------------------------------------------------------------
model.add(keras.layers.Dense(1,activation="sigmoid",input_shape=(2,)))
#-------------------------------------------------------------

model.summary()

**[2.2]** Compile the model using SGD and an appropriate loss function. Track `accuracy` as well as `loss`. Adjust the learning rate so that training converges:

In [ ]:
#-------------------------------------------------------------
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1),
              loss='binary_crossentropy',
              metrics=['accuracy'])
#-------------------------------------------------------------

The following cell fits the model to the data and plots the results:

In [ ]:
history = model.fit(dg.x, dg.t, epochs=1000, batch_size=dg.x.shape[0], verbose=0)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(history.history['loss'])
plt.grid(True)
plt.xlabel('epoch')

plt.ylabel('loss')

plt.subplot(1, 3, 2)
plt.plot(history.history['accuracy'])
plt.grid(True)
plt.xlabel('epoch')
plt.ylabel('accuracy')

xx, yy = np.meshgrid(np.arange(0, 10, 0.01), np.arange(0, 10,0.01))
xy = np.concatenate([xx.reshape([-1, 1]), yy.reshape([-1, 1])], axis=1)
z = model.predict(xy)

plt.subplot(1, 3, 3)
plt.contourf(xx, yy, z.reshape(xx.shape), 100, cmap="gray")
plt.plot(dg.x[ix0, 0], dg.x[ix0, 1], '.')
plt.plot(dg.x[ix1, 0], dg.x[ix1, 1], '.')
plt.grid(True)
plt.xlabel('height')
plt.ylabel('weight')
plt.axis([0, 10, 0, 10])
plt.legend(('t = 0', 't = 1'))

plt.show()

#### Exercise 3 [1 point]

Build a more complex neural network and test it on several cases. The network must have:

- An input layer with 8 units.
- A first hidden layer with 128 ReLU units and L1 regularization with $\lambda = 0.1$, applied only to the weights.
- A second hidden layer with 64 ReLU units.
- An output layer with 1 linear unit.
- Xavier initialization for all network weights; see [dffn.pdf](https://drive.google.com/file/d/106LPVEMFQ3Bn4THOOtP-cU-xH8ofNuBH/view?usp=sharing), page 80.

**[3.1]** Complete the following cell with your solution:

In [ ]:
model = keras.Sequential()

#-------------------------------------------------------------
model.add(keras.layers.Dense(128, activation="relu",kernel_initializer='glorot_uniform', kernel_regularizer=keras.regularizers.l1(0.1), input_dim=8))
model.add(keras.layers.Dense(64, activation="relu", kernel_initializer='glorot_uniform', name="hidden"))
model.add(keras.layers.Dense(1,kernel_initializer='glorot_uniform', name="output"))
#-------------------------------------------------------------

model.summary()

**[3.2]** If implemented correctly, the model should have 9,473 trainable parameters. Explain where this number comes from:

First layer: 128 × 8 + 128 = 1,152

Second layer: 64 × 128 + 64 = 8,256

Output layer: 1 × 64 + 1 = 65

Total: 1,152 + 8,256 + 65 = 9,473

#### Exercise 4 [1 point]

Run a series of checks to verify that the model is built correctly using the California Housing regression dataset. Details:

https://scikit-learn.org/stable/datasets/index.html#california-housing-dataset

The following cell loads and prepares the data:

In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()
x = data.data
t = data.target

**[4.1]** Split the data into training and test sets using scikit-learn's [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html). Use 30% of the data for testing and set `random_state=23` for reproducibility. Name the variables `x_train`, `x_test`, `t_train`, and `t_test`.

In [ ]:
#-------------------------------------------------------------
x_train, x_test, t_train, t_test = train_test_split(x, t,test_size=0.3,random_state=23)
x_train.shape, t_train.shape, x_test.shape, t_test.shape
#-------------------------------------------------------------

**[4.2]** Standardize the data using [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

In [ ]:
#-------------------------------------------------------------
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
#-------------------------------------------------------------

**[4.3]** Compile the model using SGD with a learning rate of `0.01` and mean squared error as the loss. Then load the weights from `random-weights`.

In [ ]:
weights_file = "MIAX5/data/california-housing/random-weights"

In [ ]:
#-------------------------------------------------------------
# TO-DO: Compile the model and load the weights here
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01), 
              loss='mse')

# Load the weights from the file above:
model.load_weights(weights_file)
#-------------------------------------------------------------

The following cell evaluates the model. Because it starts from predefined weights, the results should be reproducible in the original environment.

Expected result before training:

```text
194/194 [==============================] - 0s 1ms/step - loss: 16.3393
16.3393497467041
```

In [ ]:
model.evaluate(x_test, t_test)

As a final check, train the model for 10 epochs. Set `shuffle=False` in `model.fit` so that the data order is preserved.

Expected loss after 10 epochs in the original environment:

```text
loss: 2.1947
```

In [ ]:
_ = model.fit(x_train, t_train, epochs=10, batch_size=1000, shuffle=False)

Expected test evaluation in the original environment:

```text
194/194 [==============================] - 0s 751us/step - loss: 2.0000
2.000021457672119
```

In [ ]:
model.evaluate(x_test, t_test)

### Part II: TensorFlow

The following exercises use TensorFlow directly, without Keras.

#### Exercise 5 [1 point]

Build a linear regression model by completing the `TO-DO` blocks in `LinearRegressionModel_TF`. You do not need to calculate gradients explicitly: they can be obtained from the gradient tape.

In [ ]:
class LinearRegressionModel_TF(object):

    def __init__(self, d=2):
        # Initialize weights and bias:
        self.w = tf.Variable(tf.random.normal(shape=[d, 1], dtype=tf.dtypes.float64))  
        self.b = tf.Variable(tf.random.normal(shape=[1, 1], dtype=tf.dtypes.float64)) 
        
    def predict(self, x):
        #-----------------------------------------------------------------------
        # TO-DO block: Compute the model output y
        # Note that:
        # - x is a Nxd tensor, with N the number of patterns and d the dimension
        #   (number of features)
        # - y must be a Nx1 tensor
        #-----------------------------------------------------------------------
        y = tf.matmul(x,self.w) + self.b
        #-----------------------------------------------------------------------
        # End of TO-DO block 
        #-----------------------------------------------------------------------
        return y

    def compute_gradients(self, x, t):
        #-----------------------------------------------------------------------
        # TO-DO block: Compute the gradients db and dw of the loss function 
        # with respect to b and w
        # Note that:
        # - x is a Nxd tensor, with N the number of patterns and d the dimension
        #   (number of features)
        # - t is a Nx1 tensor
        # - y is a Nx1 tensor
        # - The gradient db (eq. dw) must have the same shape as b (eq. w) 
        #-----------------------------------------------------------------------
        with tf.GradientTape(persistent=True) as tape: 
            y = self.predict(x)
            loss = tf.reduce_mean(0.5*(y - t)*(y - t))
        db = tape.gradient(loss, self.b)
        dw = tape.gradient(loss, self.w)
        #-----------------------------------------------------------------------
        # End of TO-DO block 
        #-----------------------------------------------------------------------
        
        return db, dw
        
    def gradient_step(self, x, t, eta):
        db, dw = self.compute_gradients(x, t)
        
        #-----------------------------------------------------------------------
        # TO-DO block: Update the model parameters b and w
        #-----------------------------------------------------------------------
        new_b = self.b - eta*db 
        self.b.assign(new_b)
        new_w = self.w - eta*dw 
        self.w.assign(new_w)
        #-----------------------------------------------------------------------
        # End of TO-DO block 
        #-----------------------------------------------------------------------

    def fit(self, x, t, eta, num_iters):
        loss = np.zeros(num_iters)
        for i in range(num_iters):
            self.gradient_step(x, t, eta)
            loss[i] = self.get_loss(x, t).numpy()
        return loss

    def get_loss(self, x, t):
        y = self.predict(x)
        loss = tf.reduce_mean(0.5*(y - t)*(y - t))
        return loss

Run the following checks to verify your implementation.

Check the `predict` method:

In [ ]:
dg = dlfbt.DataGeneratorLinear(a=[-5.0, 2.0, -3.0, 2.0])
dg.create_dataset(noise=0.0, n=500)

linrm = LinearRegressionModel_TF()
linrm.w = tf.Variable(dg.a)
linrm.b = tf.Variable(dg.b)

y = linrm.predict(dg.x)
print(y.shape)                    # Should be (500, 1)
print(np.abs(y - dg.t).max())     # Should be 0 or close to 0 
print(linrm.get_loss(dg.x, dg.t)) # Should be 0 or close to 0 

Check the `compute_gradients` method:

In [ ]:
dg = dlfbt.DataGeneratorLinear(a=[-5.0, 2.0, -3.0, 2.0])
dg.create_dataset(n=50000, seed=17)

linrm = LinearRegressionModel_TF()
linrm.w = tf.Variable(dg.a)
linrm.b = tf.Variable(dg.b)

db, dw = linrm.compute_gradients(dg.x, dg.t)
print(db.shape) # Should be (1, 1)
print(db)       # Should be [[-0.00704326]] approx
print(dw.shape) # Should be (4, 1)
print(dw)       # Should be [[-0.05353578]
                #            [-0.03276935]
                #            [-0.00337341]
                #            [-0.03293776]] approx

Check the `fit` method:

In [ ]:
dg = dlfbt.DataGeneratorLinear(a=[2.0, 2.0])
dg.create_dataset(n=500, seed=17)

linrm = LinearRegressionModel_TF(2)
linrm.w = tf.Variable([[-2.0], [-2.0]], dtype=tf.dtypes.float64)
linrm.b = tf.Variable([[-3.0]], dtype=tf.dtypes.float64)

loss = linrm.fit(dg.x, dg.t, 0.01, 100)
print(loss[-1]) # Should be 2.660068435196912 approx

Plot loss against the number of iterations:

In [ ]:
plt.figure(figsize=(5, 5))
plt.plot(loss)
plt.xlabel('iteration')
plt.ylabel('loss')
plt.grid(True)
plt.show()

Plot predictions against targets; the points should lie along $y=x$:

In [ ]:
y = linrm.predict(dg.x)
plt.figure(figsize=(5, 5))
plt.plot(dg.t, y, 'o')
plt.plot([dg.t.min(), dg.t.max()], [dg.t.min(), dg.t.max()], 'r-')
plt.xlabel('real')
plt.ylabel('predicted')
plt.grid(True)
plt.show()

#### Exercise 6 [1 point]

Build a complete neural network by completing the `TO-DO` blocks in `NeuralNetwork_TF`.

In [ ]:
class NeuralNetwork_TF(object):
    #---------------------------------------------------------------------------
    # MSE loss:
    #---------------------------------------------------------------------------
    def mse_loss(y, t):
        loss = 0.5*tf.reduce_mean(tf.pow(y-t, 2.0))
        return loss
    #---------------------------------------------------------------------------
    # Cross-entropy loss:
    #---------------------------------------------------------------------------
    def cross_entropy_loss(y, t):
        loss = -tf.reduce_mean(t*tf.math.log(y) + (1.-t)*tf.math.log(1.-y))
        return loss
     
    #---------------------------------------------------------------------------
    # The constructor receives a list of tuples of the form (n, a), one tuple 
    # for each layer, where n is the number of units in the layer and a is the 
    # activation of the layer. You may use any TensorFlow activation. In 
    # particular:
    #
    # - tf.sigmoid: sigmoid activation function 
    # - tf.identity: identity activation function (to implement a linear layer) 
    # 
    # The activation function is ignored for the input layer.
    # 
    # The minimum is two layers (input and output).
    # The default value is layers=[(2, None), (1, tf.sigmoid)].
    # The output layer must have one single neuron.
    # Example of a network with 3 layers:
    # layers=[(2, None), (10, tf.relu), (1, tf.sigmoid)]
    #---------------------------------------------------------------------------
    def __init__(self, layers=[(2, None), (1, tf.sigmoid)]):
        # Network weights and activations:
        self.nlayers = len(layers) - 1
        self.W = []
        self.b = []
        self.a = []
        for l0, l1 in zip(layers[:-1], layers[1:]):
            self.W.append(tf.Variable(tf.random.normal(shape=[l1[0], l0[0]], dtype=tf.dtypes.float64)))
            self.b.append(tf.Variable(tf.random.normal(shape=[l1[0], 1], dtype=tf.dtypes.float64)))
            self.a.append(l1[1])
            
    #---------------------------------------------------------------------------
    # Implementation of the forward pass, compute the network output y given the
    # input x.
    #
    # Input: x is a tensor of dimension dxN, with N the number of patterns 
    #        and d the dimension (number of features).
    # 
    # Return: the activation y in the last layer, which must be a tensor of 
    #         dimension 1xN.
    #---------------------------------------------------------------------------
    def predict(self, x):
        #-----------------------------------------------------------------------
        # TO-DO block: loop in the network layers computing the activations. 
        # The activation of the last layer should be stored at variable y to
        # be returned.
        #-----------------------------------------------------------------------
        y = x
        for a, W, b in zip(self.a, self.W, self.b):
          z = tf.matmul(W,y) +b
          if a == tf.sigmoid:
            y = tf.sigmoid(z)
          else:
            y = tf.identity(z)
        #-----------------------------------------------------------------------
        # End of TO-DO block 
        #-----------------------------------------------------------------------
        return y
    
    #---------------------------------------------------------------------------
    # Implementation of the backward pass, compute the gradients of the loss
    # function with respect to all the weights and biases using the TensorFlow
    # GradientTape. 
    #
    # Note that the gradient tape must be persistent as many gradients must be
    # computed. 
    #
    # Input: x is a tensor of dimension dxN, with N the number of patterns 
    #        and d the dimension (number of features).
    #        t is a tensor of dimension 1xN, with N the number of patterns.
    # 
    # Return: dW is a list with the gradients of the loss function with respect 
    #         to the weights of each layer
    #         db is a list with the gradients of the loss function with respect 
    #         to the biases of each layer
    #---------------------------------------------------------------------------
    def compute_gradients(self, x, t, loss_function):
        #-----------------------------------------------------------------------
        # TO-DO block: compute the gradients db, dW
        #-----------------------------------------------------------------------
        with tf.GradientTape(persistent=True) as tape:     
            loss = self.get_loss(x,t,loss_function)     
        db = tape.gradient(loss, self.b)
        dW = tape.gradient(loss, self.W) 
        #-----------------------------------------------------------------------
        # End of TO-DO block 
        #-----------------------------------------------------------------------

        return db, dW
        
    #---------------------------------------------------------------------------
    # Gradient step:
    #---------------------------------------------------------------------------
    def gradient_step(self, x, t, eta, loss_function):
        dB, dW = self.compute_gradients(x, t, loss_function)

        #-----------------------------------------------------------------------
        # TO-DO block: Loop in layers updating the model parameters b and w
        #-----------------------------------------------------------------------     
        i=0
        for dWi in dW:
           self.W[i].assign(self.W[i] - eta*dWi)
        j=0 
        for dBj in dB:
          self.b[j].assign(self.b[j]-eta*dBj)
        #-----------------------------------------------------------------------
        # End of TO-DO block 
        #-----------------------------------------------------------------------

    #---------------------------------------------------------------------------
    # Fit the model parameters to the training data x, t.
    # Return the loss at each training epoch.
    #---------------------------------------------------------------------------
    def fit(self, x, t, eta, num_epochs, batch_size, loss_function):
        (dim, n) = x.shape
        num_batches = (n // batch_size) + ((n % batch_size) != 0)
 
        loss = np.zeros(num_epochs)
        for i in range(num_epochs):
            # Shuffle data and generate batches:
            ix = np.random.permutation(n)
            for j in range(num_batches):
                imin = j*batch_size
                imax = np.minimum((j+1)*batch_size, n)
                
                ibatch = ix[imin:imax]
                batch_x = x[:, ibatch]
                batch_t = t[:, ibatch]
                self.gradient_step(batch_x, batch_t, eta, loss_function)

            # Compute the epoch loss using all the data:
            loss[i] = self.get_loss(x, t, loss_function).numpy()
        return loss

    #---------------------------------------------------------------------------
    # Compute loss:
    #---------------------------------------------------------------------------
    def get_loss(self, x, t, loss_function):
        y = self.predict(x)
        return loss_function(y, t)

Run the following checks to verify your implementation.

The `predict` method:

In [ ]:
np.random.seed(17)
tf.random.set_seed(17)
x = np.random.randn(3, 20)

net = NeuralNetwork_TF([(3, None), (10, tf.sigmoid), (1, tf.sigmoid)])
y = net.predict(x)

# Tensor shape, should be (1, 20)
print(y.shape)

# Array values, should match those on the test file, so that all prints are 0 
# or close to 0:
with open('MIAX5/test_nn_tf_predict.pickle', 'rb') as handle:
    yexp = pickle.load(handle)
print(np.max(np.abs(y - yexp)))

The `compute_gradients` method:

In [ ]:
np.random.seed(17)
tf.random.set_seed(17)
x = np.random.randn(3, 20)
t = np.random.randn(1, 20)

net = NeuralNetwork_TF([(3, None), (10, tf.sigmoid), (1, tf.identity)])
db, dW = net.compute_gradients(x, t, NeuralNetwork_TF.mse_loss)

# Array shapes, should be:
# (10, 3)
# (10, 1)
# (1, 10)
# (1, 1)
for dwp, dbp in zip(dW, db):
    print(dwp.shape)
    print(dbp.shape)

# Array values, should match those on the test file, so that all prints are 0 
# or close to 0:
with open('MIAX5/test_nn_tf_compute_gradients.pickle', 'rb') as handle:
    [dWexp, dbexp] = pickle.load(handle)

for dwp, dbp, dwe, dbe in zip(dW, db, dWexp, dbexp):
    print(np.max(np.abs(dwp - dwe)))
    print(np.max(np.abs(dbp - dbe)))

The `fit` method for a regression problem:

In [ ]:
dg = dlfbt.DataGeneratorLinear(a=[2.0, 2.0])
dg.create_dataset(n=1000, seed=17, noise=1.0)
x = dg.x.transpose()
t = dg.t.transpose()

np.random.seed(23)
tf.random.set_seed(23)
net = NeuralNetwork_TF([(2, None), (1, tf.identity)])
loss = net.fit(x, t, 0.01, 100, 1000, NeuralNetwork_TF.mse_loss)
y = net.predict(x)

# Array shape, should be (1, 1000):
print(y.shape)

# Array values, should match those on the test file, so that all prints are 0 
# or close to 0:
with open('MIAX5/test_nn_tf_fit.pickle', 'rb') as handle:
    yexp = pickle.load(handle)
print(np.max(np.abs(y - yexp)))

Plot loss against epoch:

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(loss)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.grid(True)
plt.show()

Plot predictions against targets; the points should lie along $y=x$:

In [ ]:
y = net.predict(x).numpy().transpose()
plt.figure(figsize=(6, 6))
plt.plot(dg.t, y, 'o')
plt.plot([dg.t.min(), dg.t.max()], [dg.t.min(), dg.t.max()], 'r-')
plt.xlabel('real')
plt.ylabel('predicted')
plt.grid(True)
plt.show()

### Part III: Hyperparameter optimization

This section uses the Fashion MNIST image dataset:

https://github.com/zalandoresearch/fashion-mnist

https://keras.io/api/datasets/fashion_mnist/

Loading the data:

In [ ]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.fashion_mnist.load_data()

print(train_images.shape)
print(train_labels.shape)
print(train_labels)

print(test_images.shape)
print(test_labels.shape)
print(test_labels)

In [ ]:
train_images = train_images / 255.0
test_images = test_images / 255.0

Class names:

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

Plotting sample images:

In [ ]:
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap=plt.cm.binary)
    plt.xlabel("%d - %s" % (train_labels[i], class_names[train_labels[i]]))

Normalize the images by subtracting the mean training image:

In [ ]:
mean_img = train_images.mean(axis=0)
train_images = train_images - mean_img
test_images = test_images - mean_img

#### Exercise 7 [1 point]

Build a neural network for Fashion MNIST and perform as thorough a hyperparameter search as possible using [Keras Tuner](https://keras-team.github.io/keras-tuner/documentation/tuners/). Consider at least:

- Learning rate.
- Number of hidden layers.
- Number of units in the hidden layers.
- A regularization parameter.
- Optimizer.

Complete the indicated sections of the following cells without changing the log directory name.

In [ ]:
!pip install -q -U keras-tuner
import kerastuner as kt

**[7.1]** Define the hypermodel:

In [ ]:
def model_builder(hp):
  #-------------------------------------------------------------
  # TO-DO: Define the hyperparameters to optimize and their possible
  # values. 
  #-------------------------------------------------------------
  
  hp_hlayers = hp.Int('num_layers', 1, 10)
  hp_units = hp.Int('units',min_value=32,max_value=128,step=32),
  hp_lambda = hp.Choice('lambda', values = [0.1, 0.01, 0.001])
  hp_optimizer = hp.Choice('optimizer', values = [0, 1])
  hp_learning_rate = hp.Choice('learning_rate', values = [1.0, 0.1, 0.01, 0.001])

  model = keras.Sequential()
  #-------------------------------------------------------------
  # TO-DO: Complete the model definition by adding the required layers
  #-------------------------------------------------------------
  #Input layer
  model.add(keras.layers.Flatten(input_shape=(28, 28)))
  #Hidden layers
  for i in range(hp_hlayers):
        model.add(tf.keras.layers.Dense(units=hp.Int('units' + str(i),min_value=32,max_value=128,step=32),activation='relu',kernel_regularizer=keras.regularizers.l2(hp_lambda)))       
  #Output layer
  model.add(tf.keras.layers.Dense(10, activation='softmax'))

  if hp_optimizer == 0:
    optimizer = tf.keras.optimizers.Adam(learning_rate=hp_learning_rate)
  elif hp_optimizer == 1:
    optimizer = tf.keras.optimizers.RMSprop(learning_rate=hp_learning_rate)

  #-------------------------------------------------------------
  # TO-DO: Compile the model
  #-------------------------------------------------------------

  model.compile(optimizer=optimizer, 
                loss='sparse_categorical_crossentropy',
                metrics=['acc'])
  return model

Deleting the log directory:

In [ ]:
!rm -rf training-logs/keras_tuner

Create a `Hyperband` tuner using the hypermodel above. Optimize validation accuracy with a maximum of 20 epochs.

In [ ]:
tuner = kt.Hyperband(model_builder,
                     objective = 'val_acc', 
                     max_epochs = 20,
                     factor = 3,
                     directory = 'training-logs/keras_tuner',
                     project_name = 'kt')  

Summarize the search space and check that it contains all candidate values for each hyperparameter.

In [ ]:
tuner.search_space_summary()

**[7.2]** Randomly divide the training data into two subsets. Use 50,000 examples in `(tr_images, tr_labels)` to train the hypermodels and 10,000 examples in `(val_images, val_labels)` for validation. You can use [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) from `sklearn.model_selection`.

In [ ]:
from sklearn.model_selection import train_test_split

#-------------------------------------------------------------
# TO-DO: Create the data subsets (tr_images, tr_labels) and 
# (val_images, val_labels)

tr_images, val_images, tr_labels, val_labels = train_test_split(train_images,
                                                    train_labels,
                                                    test_size=1/6,
                                                    stratify=train_labels)
#-------------------------------------------------------------

Hyperparameter search:

In [ ]:
tuner.search(tr_images, tr_labels,
             epochs=20,
             validation_data=(val_images, val_labels))

**[7.3]** Select the best hyperparameters, retrain the model on all training data, and apply it to the test set.

In [ ]:
#-------------------------------------------------------------
# TO-DO: Select the best hyperparameters, retrain the model on all
# training data, and apply it to the test set
best_hps = tuner.get_best_hyperparameters()[0]
best_hps.values

model = tuner.hypermodel.build(best_hps)
model.fit(train_images, train_labels, epochs = 20, validation_data = (test_images, test_labels))
#-------------------------------------------------------------

#### Exercise 8 [1 point]

Build a neural network for Fashion MNIST and perform as thorough a hyperparameter search as possible using TensorBoard's [HParams Dashboard](https://www.tensorflow.org/tensorboard/hyperparameter_tuning_with_hparams) with 5-fold cross-validation. Consider at least:

- Learning rate.
- Number of hidden layers.
- Number of units in the hidden layers.
- A regularization parameter.
- Optimizer.

Complete the indicated sections of the following cells.

In [ ]:
from tensorboard.plugins.hparams import api as hp
from sklearn.model_selection import StratifiedKFold

**[8.1]** Define the hypermodel. Use 5-fold cross-validation and calculate `accuracy` as the average across the five validation folds.

In [ ]:
def train_test_model(hparams):

  # 5-fold cross-validation
  kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)

  # Loop over folds:
  mean_accuracy = 0.0
  for itrain, itest in kfold.split(train_images, train_labels):
    model = keras.Sequential()
    #-------------------------------------------------------------
    # TO-DO: Complete the model definition by adding the required layers
    #-------------------------------------------------------------
   
    model.add(keras.layers.Flatten(input_shape=(28, 28)))
    for i in range(hparams['num_hlayers']):
      model.add(keras.layers.Dense(hparams['num_units'], activation="relu", kernel_regularizer=keras.regularizers.l2(hparams['lambda'])))
    model.add(keras.layers.Dense(10, activation="softmax"))

    if hparams['optimizer'] == "adam":
      optimizer = tf.keras.optimizers.Adam(learning_rate=hparams['learning_rate'])
    elif hparams['optimizer'] == "sgd":
      optimizer = tf.keras.optimizers.SGD(learning_rate=hparams['learning_rate'])
    #-------------------------------------------------------------
    # TO-DO: Compile the model
    #-------------------------------------------------------------
    model.compile(optimizer=hparams['optimizer'], 
                loss='sparse_categorical_crossentropy',
                metrics=['acc'])
    #-------------------------------------------------------------
    # TO-DO: Train the model using the itrain indices
    #-------------------------------------------------------------
    history = model.fit(train_images[itrain], 
                        train_labels[itrain], 
                        epochs=10,
                        verbose=0,
                        batch_size=50)    
    #-------------------------------------------------------------
    # TO-DO: Evaluate the model using the itest indices and calculate accuracy
    #-------------------------------------------------------------
    loss, acc = model.evaluate(train_images[itest], train_labels[itest], verbose=0)


    # Accumulate accuracy:
    mean_accuracy += acc
  
  return mean_accuracy/5.0

Function for running hypermodel trials:

In [ ]:
def run(run_dir, hparams):
  with tf.summary.create_file_writer(run_dir).as_default():
    hp.hparams(hparams) 
    accuracy = train_test_model(hparams)
    tf.summary.scalar('accuracy', accuracy, step=1)

Deleting the log directory:

In [ ]:
!rm -rf training-logs/hparam

**[8.2]** Run the hypermodels in a loop:

In [ ]:
session_num = 0

for num_hlayers in [1, 2]:
  for num_units in [16, 32, 64, 128]:
    for optimizer in ['adam', 'sgd']:
      for learning_rate in [1.0, 0.1, 0.01, 0.001]:
        for llambda in [0.001, 0.0001]:
               
          hparams = {
          'num_units': num_units,
          'learning_rate': learning_rate,
          'optimizer': optimizer,
          'num_hlayers': num_hlayers,
          'lambda':llambda,
          }
          run_name = "run-%d" % session_num
          print('--- Starting trial: %s' % run_name)
          print(hparams)
          run('training-logs/hparam/' + run_name, hparams)
          session_num += 1

Original exercise instruction: use [TensorBoard.dev](https://tensorboard.dev/) to upload the generated logs to the cloud, authorizing Google to publish them in your account. This is a historical instruction; review compatibility before running it.

In [ ]:
!tensorboard dev upload --logdir training-logs/hparam --name "Exercise 8" \
    --description "Hyperparameter comparison"

In the original workflow, the preceding cell returned a link to the TensorBoard trial results. Copy that link below so that the trials can be reviewed.

In [ ]:
#-------------------------------------------------------------
# TO-DO: Copy your TensorBoard link here, for example
https://tensorboard.dev/experiment/G8Z4w3QdRB2z1i2crlOziw/

#-------------------------------------------------------------

**[8.3]** Select the best hyperparameters, retrain the model on all training data, and apply it to the test set.

In [ ]:
#-------------------------------------------------------------
# TO-DO: Select the best hyperparameters, retrain the model on all
# training data, and apply it to the test set
#-------------------------------------------------------------

# After filtering in TensorBoard, the best hyperparameters are:
# num_units: 128
# learning_rate: 0.01
# optimizer: adam
# num_hlayers: 2
# lambda: 0.0001


model = keras.Sequential()
model.add(keras.layers.Flatten(input_shape=(28, 28)))
model.add(keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(0.0001)))
model.add(keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(0.0001)))
model.add(keras.layers.Dense(10, activation="softmax"))

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01), 
    loss='sparse_categorical_crossentropy', 
    metrics=['acc'],
)

# Train the model
    
history = model.fit(train_images, 
                    train_labels, 
                    epochs=10, 
                    batch_size=50)

loss, acc = model.evaluate(test_images, test_labels, verbose=0)
print(acc)  
#-------------------------------------------------------------


### Part IV: Neural networks applied to a real-world problem

Choose a problem and apply what you have learned. Solve it using a neural network and optimize its hyperparameters. Assessment criteria include:

- Complexity of the chosen problem.
- Problem description and analysis.
- Data preparation appropriate to the objectives.
- Modeling strategy, network selection, hyperparameter selection, and validation.
- Results.
- Final conclusions.

#### Exercise 9 [1 point]

- Describe the problem, objectives, and data. Explain how the data is prepared for the neural network.
- Include code cells to support your explanations if needed.

Build an RNN model using LSTM layers to predict the price or trend of a stock, in this case Google stock.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

**Loading and preparing the data**

The original notebook identifies Yahoo Finance as the source of the data and uses five years of Google stock history:

https://finance.yahoo.com/quote/GOOG/history?p=GOOG

Reminder: place `GOOG.csv` in the working directory.

In [ ]:
data=pd.read_csv("GOOG.csv")
data

In [ ]:
data.shape

**Training/test split**

The target variable is the closing price, `Close`.

In [ ]:
real_data = data.iloc[:, 4:5]
real_data

Split the data into training and test sets.

In [ ]:
train = data.iloc[:800, 4:5].values
test = data.iloc[800:, 4:5].values

In [ ]:
#Reshape the test data to the required dimensions
test_2 = real_data[len(real_data) - len(test) - 60:].values
test_2 = test_2.reshape(-1,1)

**Normalizing training and test data**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
sc = MinMaxScaler(feature_range = (0, 1))

In [ ]:
train_scaled = sc.fit_transform(train)
train_scaled.shape

In [ ]:
test_scaled = sc.fit_transform(test_2)
test_scaled.shape

**Transforming the data**

Transform the training and test data into three-dimensional arrays, the input format required by the LSTM. Each input sequence contains 60 daily time steps with one closing-price feature, and the model predicts one output. Apply this procedure to both training and test data.

In [ ]:
timestep = 60

In [ ]:
# Create the training dataset
X_train = []
y_train = []
for i in range(60, 800):
    X_train.append(train_scaled[i-timestep:i, 0])
    y_train.append(train_scaled[i, 0])

In [ ]:
# Create the test dataset
X_test = []
y_test = []
for i in range(60, 519):
    X_test.append(test_scaled[i-timestep:i, 0])
    y_test.append(test_scaled[i, 0])

In [ ]:
# Load the variables
X_train, y_train = np.array(X_train), np.array(y_train)
X_test, y_test = np.array(X_test), np.array(y_test)

In [ ]:
# Reshape the data to the dimensions required by the model
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

In [ ]:
# Check the training dataset dimensions
X_train.shape

In [ ]:
# Check the test dataset dimensions
X_test.shape

#### Exercise 10 [1 point]

- Describe how the network is trained, including the evaluation approach and loss function.
- Describe the selected model and the validation procedure used to determine its architecture and hyperparameters.
- Describe the results and draw conclusions.
- Include code cells to support your explanations if needed.

**Building the model**

The network has the following architecture:

- An initial LSTM layer with 50 units, followed by dropout with a rate of 0.2 to help prevent overfitting.
- Three further LSTM layers with 50 units each, each followed by dropout with a rate of 0.2.
- A dense output layer with 1 unit and a linear activation for regression.

The LSTM layers use their default settings. The original write-up describes Xavier weight initialization and a `tanh` activation; see the layer implementation for the actual defaults used by the installed library.

In [ ]:
%tensorflow_version 2.x
import tensorflow as tf
from tensorflow import keras

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM,Dense,Dropout

In [ ]:
model = Sequential()

Provide the previous 60 days of data to the neural network for each prediction.

In [ ]:
#Input layer
model.add(LSTM(units = 50, return_sequences = True, input_shape = (X_train.shape[1], 1)))
model.add(Dropout(0.2))

In [ ]:
#Hidden layers
model.add(LSTM(units = 50, return_sequences = True))
model.add(Dropout(0.2))
model.add(LSTM(units = 50, return_sequences = True))
model.add(Dropout(0.2))
model.add(LSTM(units = 50))
model.add(Dropout(0.2))

In [ ]:
#Output layer
model.add(Dense(units = 1))

In [ ]:
model.summary()

After building the model, configure training with `compile()`. Use the Adam optimizer with a learning rate of 0.01 and mean squared error (MSE) as the loss function.

In [ ]:
model.compile(optimizer =tf.keras.optimizers.Adam(learning_rate=0.01),
              loss = 'mean_squared_error')

Train the model with `fit()`, specifying:

- Training data.
- Number of epochs.

Train for 20 epochs.

In [ ]:
nepochs = 20
history = model.fit(X_train, y_train,
          epochs = nepochs)

The original experiment reported a substantial decrease in loss across epochs, with some noise. The following cell plots the loss history:

In [ ]:
hd = history.history

epochs = range(1, nepochs+1)

plt.figure(figsize=(12,6))

plt.subplot(1,2,2)
plt.plot(epochs, hd['loss'], "r")
plt.grid(True)
plt.xlabel("epoch")
plt.ylabel("loss")


plt.show()

**Hyperparameter optimization**

Use Keras Tuner to search over the learning rate, the number of units in the hidden layers, and the dropout rate.

In [ ]:
!pip install -q -U keras-tuner
import kerastuner as kt

Defining the hypermodel:

In [ ]:
def model_builder(hp):

  hp_learning_rate = hp.Choice('learning_rate', values = [1.0, 0.1, 0.01])
  hp_units = hp.Int('units', min_value = 50, max_value = 200, step = 50)
  hp_dropout_rate = hp.Float('dropout_rate', min_value = 0.1, max_value = 0.5, step = 0.1)
  
  model = keras.Sequential()

  #Input layer
  model.add(LSTM(units = 50, return_sequences = True, input_shape = (X_train.shape[1], 1)))
  model.add(Dropout(0.2))

  #Hidden layers
  model.add(LSTM(units = hp_units, return_sequences = True))
  model.add(Dropout(hp_dropout_rate))
  model.add(LSTM(units = hp_units, return_sequences = True))
  model.add(Dropout(hp_dropout_rate))
  model.add(LSTM(units = hp_units))
  model.add(Dropout(hp_dropout_rate))

  #Output layer
  model.add(Dense(units = 1))

  model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate), 
              loss='mean_squared_error')
  
  return model

Create a Hyperband tuner using the hypermodel above. Optimize validation loss with a maximum of 20 epochs.

In [ ]:
!rm -rf my_dir/exercises_9_and_10/

In [ ]:
tuner = kt.Hyperband(model_builder,
                     objective = 'val_loss', 
                     max_epochs = 20,                     
                     factor = 3,
                     directory = 'my_dir',
                     project_name = 'exercises_9_and_10')

Summarize the search space and check that it contains all candidate values for each hyperparameter.

In [ ]:
tuner.search_space_summary()

Hyperparameter search:

In [ ]:
tuner.search(X_train, y_train,
             epochs=20,
             validation_data=(X_test, y_test))

Select the best hyperparameters:

In [ ]:
best_hps = tuner.get_best_hyperparameters()[0]
best_hps.values

Retrain the model with the best hyperparameters:

In [ ]:
model = tuner.hypermodel.build(best_hps)
model.fit(X_train, y_train, epochs = nepochs, validation_data = (X_test, y_test))

The `predict()` method returns the predictions:

In [ ]:
predictions = model.predict(X_test)
predictions = sc.inverse_transform(predictions)

Plot the predictions against the actual test prices:

In [ ]:
plt.figure(figsize=(20,10))
plt.plot(data.loc[800:, 'Date'],test , color = 'red', label = 'Actual price')
plt.plot(data.loc[800:, 'Date'],predictions, color = 'blue', label = 'Prediction')
plt.xticks(np.arange(0,459,50))
plt.xlabel('Time')
plt.ylabel('Actual price')
plt.legend()
plt.show()

The original write-up reports that the model followed the stock trend with some lag. This visual observation is not sufficient to establish out-of-sample predictive performance; see the methodological limitations in README.md.